# Van Udyan Plant Species Classification: Deep Learning Training & Evaluation

**Project:** Deep Learning-Based Plant Species Classification for Urban Plantation Monitoring  
**Field Organization:** Reform Social Welfare Foundation (RSWF), Pune - Van Udyan Biodiversity Monitoring  
**GitHub:** [https://github.com/ppriyansh-05/DL-Plant-Species-Classification-for-van-udhyan-](https://github.com/ppriyansh-05/DL-Plant-Species-Classification-for-van-udhyan-)  

---
### Quick Run Instructions in Google Colab:
1. **Set GPU Accelerator**: Go to **Runtime -> Change runtime type -> Hardware accelerator: T4 GPU**.
2. **Upload / Mount Dataset**:
   - **Option A (Direct ZIP Upload)**: Upload `van_udyan_cleaned_dataset.zip` to `/content/` via the file panel, then run Cell 2.
   - **Option B (Google Drive Mount)**: If your dataset or NGO Drive folder is mounted, set `USE_GOOGLE_DRIVE = True` in Cell 2.
3. **Run All Cells**: Click **Runtime -> Run all** (`Ctrl+F9`).
4. All artifacts (best model `.keras`, confusion matrix, classification report, test predictions) will be automatically evaluated and saved to Google Drive / local disk.

In [ ]:
# Cell 1: Environment & GPU Verification
import os, sys, shutil, random, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support

print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"[GPU Active] Found {len(gpus)} GPU device(s): {gpus}")
else:
    print("[Notice] No GPU detected. Running on CPU may be slow.")

# Fix random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


In [ ]:
# Cell 2: Dataset Access & Extraction
USE_GOOGLE_DRIVE = False  # Set to True if mounting from Google Drive
DRIVE_ZIP_PATH = "/content/drive/MyDrive/van_udyan_cleaned_dataset.zip"
LOCAL_ZIP_PATH = "/content/van_udyan_cleaned_dataset.zip"
TARGET_EXTRACT_DIR = Path("/content/data/processed")

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    zip_to_use = DRIVE_ZIP_PATH
else:
    zip_to_use = LOCAL_ZIP_PATH

if not TARGET_EXTRACT_DIR.exists() or len(list(TARGET_EXTRACT_DIR.glob('*/*.*'))) == 0:
    if Path(zip_to_use).exists():
        print(f"Extracting dataset from {zip_to_use}...")
        import zipfile
        with zipfile.ZipFile(zip_to_use, 'r') as zip_ref:
            zip_ref.extractall(TARGET_EXTRACT_DIR)
        print(f"Extraction complete: {TARGET_EXTRACT_DIR}")
    elif Path("van_udyan_cleaned_dataset.zip").exists():
        import zipfile
        with zipfile.ZipFile("van_udyan_cleaned_dataset.zip", 'r') as zip_ref:
            zip_ref.extractall(TARGET_EXTRACT_DIR)
        print("Extracted local van_udyan_cleaned_dataset.zip successfully.")
    else:
        print(f"[Warning] Neither {zip_to_use} nor local zip found. Please upload van_udyan_cleaned_dataset.zip to /content/.")
else:
    print(f"Target directory already populated: {TARGET_EXTRACT_DIR}")


In [ ]:
# Cell 3: Dataset Integrity & Class Distribution Verification
DATA_DIR = TARGET_EXTRACT_DIR
TRAIN_DIR = DATA_DIR / "train"
VAL_DIR = DATA_DIR / "val"
TEST_DIR = DATA_DIR / "test"
BENCHMARK_DIR = DATA_DIR / "test_van_udyan_benchmark"

IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32

for name, p in [('Train', TRAIN_DIR), ('Val', VAL_DIR), ('Test', TEST_DIR), ('Benchmark', BENCHMARK_DIR)]:
    if p.exists():
        classes = sorted([d.name for d in p.iterdir() if d.is_dir()])
        images = list(p.glob("*/*.*"))
        print(f"{name:10s}: {len(classes)} classes, {len(images)} total images")
    else:
        print(f"{name:10s}: Directory not found ({p})")

metadata_file = DATA_DIR / "metadata.csv"
if metadata_file.exists():
    df_meta = pd.read_csv(metadata_file)
    print(f"Master metadata loaded: {len(df_meta)} rows across {df_meta['species'].nunique()} species.")


In [ ]:
# Cell 4: Data Loaders with Auto-Stepping (No Input-Ran-Out-Of-Data Warnings)
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels='inferred',
    label_mode='categorical',
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    labels='inferred',
    label_mode='categorical',
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    labels='inferred',
    label_mode='categorical',
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print(f"\nTotal Target Classes: {NUM_CLASSES}")
print("Sample Classes:", class_names[:5])

# Compute class weights to address residual imbalance
from sklearn.utils.class_weight import compute_class_weight
y_train_indices = []
for _, labels in train_ds:
    y_train_indices.extend(np.argmax(labels.numpy(), axis=1))
classes_unique = np.unique(y_train_indices)
class_weights_arr = compute_class_weight('balanced', classes=classes_unique, y=y_train_indices)
class_weights = dict(zip(classes_unique, class_weights_arr))
print(f"Class weights computed for {len(class_weights)} classes.")

# Pipeline optimization
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)


In [ ]:
# Cell 5: Training-Only Augmentation Layer
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.12),
    layers.RandomContrast(0.1),
], name="training_augmentation")

# Preview sample augmentations
plt.figure(figsize=(12, 5))
for images, _ in train_ds.take(1):
    sample = images[0]
    for i in range(6):
        ax = plt.subplot(2, 3, i + 1)
        aug = data_augmentation(tf.expand_dims(sample, 0))
        plt.imshow(aug[0].numpy().astype("uint8"))
        plt.axis("off")
plt.suptitle("Sample Training Augmentations")
plt.tight_layout()
plt.show()


In [ ]:
# Cell 6: Model Architecture (MobileNetV2 Transfer Learning)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False  # Freeze feature extractor for Stage 1

inputs = layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3), name="input_image")
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D(name="gap")(x)
x = layers.BatchNormalization(name="batch_norm")(x)
x = layers.Dense(256, activation="relu", name="dense_head")(x)
x = layers.Dropout(0.40, name="dropout")(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="predictions")(x)

model = keras.Model(inputs=inputs, outputs=outputs, name="VanUdyan_MobileNetV2")
model.summary()


In [ ]:
# Cell 7: Compile and Setup Callbacks
CHECKPOINT_PATH = "best_van_udyan_mobilenetv2.keras"

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy", keras.metrics.TopKCategoricalAccuracy(k=3, name="top_3_accuracy")]
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        CHECKPOINT_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]


In [ ]:
# Cell 8: Stage 1 Training (Feature Extractor)
EPOCHS_STAGE1 = 15
print("Starting Stage 1: Feature Extraction...")
# Notice: steps_per_epoch is omitted to let Keras automatically step through all batches cleanly without exhausting
history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_STAGE1,
    class_weight=class_weights,
    callbacks=callbacks
)


In [ ]:
# Cell 9: Stage 2 Training (Controlled Fine-Tuning)
print("Unfreezing top layers for Stage 2 Fine-Tuning...")
base_model.trainable = True
# Freeze the lower 100 layers and fine-tune remaining top convolutional bottlenecks
for layer in base_model.layers[:100]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy", keras.metrics.TopKCategoricalAccuracy(k=3, name="top_3_accuracy")]
)

EPOCHS_STAGE2 = 12
history_stage2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_STAGE2,
    class_weight=class_weights,
    callbacks=callbacks
)


In [ ]:
# Cell 10: Quantitative Evaluation on Held-Out Test Set
print(f"Loading best model checkpoint from {CHECKPOINT_PATH}...")
best_model = keras.models.load_model(CHECKPOINT_PATH)

test_eval = best_model.evaluate(test_ds)
print(f"\n--- Held-Out Test Split Metrics ---")
print(f"Test Loss:            {test_eval[0]:.4f}")
print(f"Test Top-1 Accuracy:  {test_eval[1] * 100:.2f}%")
print(f"Test Top-3 Accuracy:  {test_eval[2] * 100:.2f}%")

# Collect predictions across all test batches
y_true = []
y_pred_probs = []
for images, labels in test_ds:
    preds = best_model.predict(images, verbose=0)
    y_pred_probs.extend(preds)
    y_true.extend(np.argmax(labels.numpy(), axis=1))

y_true = np.array(y_true)
y_pred_probs = np.array(y_pred_probs)
y_pred = np.argmax(y_pred_probs, axis=1)

report = classification_report(y_true, y_pred, target_names=class_names, digits=4)
print("\nDetailed Classification Report:")
print(report)

with open("evaluation_classification_report.txt", "w") as f:
    f.write(report)


In [ ]:
# Cell 11: Confusion Matrix Diagnostics & Training Curves
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(20, 18))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.title(f"Confusion Matrix - MobileNetV2 (38 Classes) - Test Accuracy: {test_eval[1]*100:.2f}%")
plt.xlabel("Predicted Class")
plt.ylabel("Ground Truth Class")
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=300)
plt.show()

# Save ordered class mapping JSON
import json
with open("class_mapping.json", "w") as f:
    json.dump({i: name for i, name in enumerate(class_names)}, f, indent=2)
print("Saved class_mapping.json and confusion_matrix.png successfully.")


In [ ]:
# Cell 12: Dedicated Evaluation on Pure Van Udyan Local In-Situ Benchmark
if BENCHMARK_DIR.exists() and len(list(BENCHMARK_DIR.glob('*/*.*'))) > 0:
    print("Evaluating best model exclusively on authentic in-situ Van Udyan field photographs...")
    bench_ds = tf.keras.utils.image_dataset_from_directory(
        BENCHMARK_DIR,
        labels='inferred',
        label_mode='categorical',
        image_size=(IMG_HEIGHT, IMG_WIDTH),
        batch_size=BATCH_SIZE,
        shuffle=False
    )
    bench_eval = best_model.evaluate(bench_ds)
    print(f"\n--- Dedicated In-Situ Van Udyan Benchmark Results ---")
    print(f"Benchmark Top-1 Accuracy: {bench_eval[1] * 100:.2f}%")
    print(f"Benchmark Top-3 Accuracy: {bench_eval[2] * 100:.2f}%")
else:
    print("No benchmark photos found at", BENCHMARK_DIR)


In [ ]:
# Cell 13: Export Final Artifacts to Google Drive (Optional)
EXPORT_TO_DRIVE = False
DRIVE_EXPORT_FOLDER = "/content/drive/MyDrive/Van_Udyan_Model_Results"

if EXPORT_TO_DRIVE:
    os.makedirs(DRIVE_EXPORT_FOLDER, exist_ok=True)
    artifacts = [CHECKPOINT_PATH, "confusion_matrix.png", "evaluation_classification_report.txt", "class_mapping.json"]
    for art in artifacts:
        if os.path.exists(art):
            shutil.copy2(art, DRIVE_EXPORT_FOLDER)
            print(f"Exported {art} -> {DRIVE_EXPORT_FOLDER}")
else:
    print("Artifacts saved locally in /content/. Set EXPORT_TO_DRIVE=True to back up directly to Google Drive.")
